In [0]:
 # Same pinned set as s05_oot / s09 (the project has no requirements.lock yet)
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" openpyxl --quiet
dbutils.library.restartPython()

In [0]:
# s06b (no-FTS variant): cut-off analysis for the locked no-FTS model 1.0.1-nofts, Models A and C
# (spec 8.2, 8.3, 9.1-9.3; E13 scoring code). Mirrors the FTS cut-off analysis on cc_eda with the same definitions.
# REPORT ONLY. APPROVED APPLICANTS ONLY (acard_master, read-only). Writes only under <artifacts_root>/<cutoff_analysis.output_folder>/
# plus the requested segment entry in reports/open_questions.md. Never writes under main_artifacts_root or the FTS model folder.
# Owner decisions: approval base = cutoff_analysis.approval_base_statuses (EXCLUDED left out); bad rates = bad_rate_statuses
# (GOOD + BAD); INDETERMINATE / IMMATURE / EXCLUDED_PERF reported as separate counts; segments = WATCH variables only.
# Benchmarks (2026-11-18): an entry with a table is read from that table (read-only) and joined on app_id = CRNNO, or
# CREDITCARDNO where the CRNNO is repeated in acard_master (same rule as s00_build_a_score_snapshot); A_SCORE is used only if
# score month - application month is 0 or 1 (same rule as the FTS run).
import sys, os, json, time, pickle, shutil, tempfile, hashlib, importlib, warnings, re
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import spearmanr, chi2_contingency
from sklearn.metrics import roc_auc_score
from pyspark.sql import functions as F

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
warnings.filterwarnings("ignore", message="Missing values detected")

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)
CA = cfg["cutoff_analysis"]
MODELS_USED = list(CA["models"])
assert MODELS_USED == ["A", "C"], f"this module scores Models A and C only, config has {MODELS_USED}"
assert CA["performance_window"] == "OOT", "config.cutoff_analysis.performance_window must be OOT"
GRID = [float(r) for r in CA["approval_rate_grid"]]
BW = int(CA["score_band_width"])
BASE_STATUSES = list(CA["approval_base_statuses"])
LABELLED = list(CA["bad_rate_statuses"])
assert set(LABELLED) == set(cfg["labelled_statuses"]) == {"GOOD", "BAD"}, f"bad_rate_statuses must be GOOD + BAD, got {LABELLED}"
assert set(BASE_STATUSES) == {"GOOD", "BAD", "INDETERMINATE", "IMMATURE", "EXCLUDED_PERF"}, f"unexpected approval_base_statuses {BASE_STATUSES}"
OTHER = ["INDETERMINATE", "IMMATURE", "EXCLUDED_PERF"]       # in the approval base, never in bad rates

# Benchmarks (spec 8.3): config.cutoff_analysis.compare_with = list of {name, column, higher_is_lower_risk, role[, table, source_table, join_key, definition]}
BENCH_DESC = {"A_SCORE": "current decision score",
              "AECB_SCORE": "bureau score, reference only (pre-upgrade version in OOT; not a model input)"}
ASCORE_OFFSETS = [0, 1]                 # owner 2026-11-18: same score-month rule as the FTS run (table entries with a score date)
BENCHES = []
for e in CA.get("compare_with") or []:
    assert {"name", "column", "higher_is_lower_risk"} <= set(e), f"compare_with entry needs name, column, higher_is_lower_risk: {e}"
    if e.get("table"):
        assert str(e.get("join_key")) == "app_id", f"{e['name']}: join_key must be app_id"
    k = re.sub(r"\W", "_", str(e["name"])).upper()
    assert k not in MODELS_USED and not k.startswith("FTS_")
    BENCHES.append({**e, "key": k, "desc": BENCH_DESC.get(k, str(e.get("role", "")))})
BENCH_KEYS = [b["key"] for b in BENCHES]
BENCH_NAME_OF = {b["key"]: str(b["name"]) for b in BENCHES}

FTS_CFG = CA.get("compare_with_fts") or {}
FTS_ON = bool(FTS_CFG.get("enabled"))
FTS_REG_PATH = FTS_CFG.get("registry") if FTS_ON else None
if FTS_ON:
    assert list(FTS_CFG["models"]) == MODELS_USED, "compare_with_fts.models must be the same models (A, C)"
FTS_FLAG_NAME = "FTS_Ref_No"            # has_fts_data = flag column non-null and non-empty after trim (segment only)

ARTIFACTS = common.artifacts_path(cfg)
REG_PATH = os.path.join(ARTIFACTS, "model_registry.json")
OUT = os.path.join(ARTIFACTS, CA["output_folder"])
MAIN_ROOT = "/" + str(cfg["main_artifacts_root"]).lstrip("/")
FTS_ROOT = os.path.dirname(FTS_REG_PATH) if FTS_ON else None
for _forbidden in [MAIN_ROOT, FTS_ROOT]:
    if _forbidden:
        assert not (os.path.abspath(OUT) + os.sep).startswith(os.path.abspath(_forbidden) + os.sep), f"output folder is under {_forbidden}"
os.makedirs(os.path.join(OUT, "run_summaries"), exist_ok=True)
VERSION = str(cfg["registry"]["version"])
assert VERSION == "1.0.1-nofts", f"this module is for the no-FTS variant 1.0.1-nofts, config.registry.version = {VERSION}"
XLSX_PATH = os.path.join(OUT, f"cutoff_analysis_v{VERSION.replace('-', '_')}_A_C.xlsx")    # cutoff_analysis_v1.0.1_nofts_A_C.xlsx
SUMMARY_PATH = os.path.join(OUT, "run_summaries", "s06b_cutoff_analysis.json")
OQ_PATH = os.path.join(ARTIFACTS, "reports", "open_questions.md")    # owner request: the only write outside OUT

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.{cfg['tables']['master']}"
E13_STORED = f"{cat}.{sch}.{cfg['tables']['oot_predictions']}"   # no-FTS E13 case-level OOT scores (spot check only)
E13_STORED_FTS = f"{cat}.{sch}.acard_oot_predictions"             # FTS v1.0.1 E13 case-level OOT scores (spot check only)
id_col, DATE_A, DECISION_A = cfg["columns"]["id"], cfg["columns"]["date"], cfg["columns"]["decision_time"]
BOOT = cfg["selection"]["bootstrap"]
SPLITS = cfg["oot_report_splits"]
WATCH = list(cfg["column_roles"]["watch"])
SEGMENTS = cfg["columns"].get("segments") or []
EPS = cfg["funnel"]["L3"]["eps"]

CAVEAT = ("Approval rates are relative to previously approved applicants, not to all applicants; "
          "they overstate the through-the-door approval rate.")

# Report-only constants (same as the FTS run; nothing is selected, dropped or fitted with them)
SEG_MIN_BADS = 50                      # STEP 4.4: segments with < 50 bads -> bad rate and count only (no CI)
AGE_BANDS = [0, 25, 30, 35, 40, 45, 50, 60, 200]
TOP_N_WATCH = 10                       # other categories -> OTHER
IMM_RATIO_FLAG = 1.5                   # IMMATURE flag: rate ratio vs the comparison group
IMM_BAND_MIN_N = 30                    # IMMATURE flag: minimum band size
IMM_P_FLAG = 0.05                      # IMMATURE flag: chi-square p-value

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
RUN_TIME = datetime.now().isoformat(timespec="seconds")
_norm = common._norm
WRITTEN = []

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def _json_default(o):
    if hasattr(o, "item"):
        return o.item()
    if isinstance(o, float) and np.isnan(o):
        return None
    return str(o)

def _guard(dst):
    ok = os.path.abspath(dst).startswith(os.path.abspath(OUT) + os.sep) or os.path.abspath(dst) == os.path.abspath(OQ_PATH)
    assert ok, f"refusing to write outside {OUT}: {dst}"

def write_text(text, dst):
    """Local temp file, then copy to the UC volume (copyfile: FUSE does not allow chmod/utime)."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s06b_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        fh.write(text)
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

def write_json(obj, dst):
    write_text(json.dumps(obj, indent=2, default=_json_default), dst)

def save_excel(sheets, dst):
    """sheets: name -> DataFrame, or list of (title, DataFrame) blocks. Row 1 of EVERY sheet is the approved-only caveat."""
    _guard(dst)
    tmp = os.path.join(tempfile.mkdtemp(prefix="s06b_"), os.path.basename(dst))
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, obj in sheets.items():
            assert len(name) <= 31, f"sheet name too long: {name}"
            blocks = obj if isinstance(obj, list) else [(None, obj)]
            pd.DataFrame(columns=[CAVEAT]).to_excel(xw, sheet_name=name, startrow=0, index=False)
            r = 2
            for title, df in blocks:
                df = df.copy()
                for c in df.columns:
                    if isinstance(df[c].dtype, pd.PeriodDtype) or str(df[c].dtype) in ("string", "category") or str(df[c].dtype).startswith("interval"):
                        df[c] = df[c].astype(str)
                if title:
                    pd.DataFrame(columns=[title]).to_excel(xw, sheet_name=name, startrow=r, index=False)
                    r += 1
                df.to_excel(xw, sheet_name=name, startrow=r, index=False)
                r += len(df) + 3
    shutil.copyfile(tmp, dst)
    WRITTEN.append(dst)

_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}\nOutput      : {OUT}\nWorkbook    : {XLSX_PATH}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | grid {GRID} | band width {BW}")
print(f"Benchmarks  : {[(b['name'], b.get('table') or 'acard_master', b['column']) for b in BENCHES]} | FTS comparison: {FTS_ON} ({FTS_REG_PATH})")

In [0]:
def load_registry(reg_path, label):
    """Verify a locked registry (sign-off, every file hash, Platt file hash) and load Models A and C. Read-only."""
    assert os.path.exists(reg_path), f"ABORT: {label} registry not found: {reg_path}"
    reg = read_json(reg_path)
    assert reg.get("locked") is True, f"ABORT: {label} registry is not locked"
    for k in ("decision_rules_signed_by", "signed_date"):
        assert str(reg.get("signoff", {}).get(k) or "").strip(), f"ABORT: {label} sign-off field {k} is empty"
    files = {k: v["path"] for k, v in reg["files"].items()}
    locked_sha = {k: v["sha256"] for k, v in reg["files"].items()}
    bad = [k for k, p in files.items() if not os.path.exists(p) or sha256_file(p) != locked_sha[k]]
    assert not bad, f"ABORT: {label}: {len(bad)} locked model/binning/fold/OOF file(s) changed since the lock: {bad}"
    need = ["models/A/binning_process.pkl", "models/A/lr_sklearn.pkl", "models/C/lgbm.txt", "models/model_spec.json", "oof_predictions.parquet"]
    assert not [k for k in need if k not in files], f"ABORT: {label} registry lacks {[k for k in need if k not in files]}"
    cal_ref = reg["oot_evaluation"]["calibration_file"]
    platt_sha = sha256_file(cal_ref["path"])
    assert platt_sha == cal_ref["sha256"], f"ABORT: {label} Platt calibration file differs from the hash in the registry"
    doc = read_json(cal_ref["path"])
    assert str(doc["registry_version"]) == str(reg["version"]) and doc["method"] == "platt" and doc["fit_on"] == "dev_oof", \
        f"ABORT: {label} Platt file is not the DEV-OOF Platt fit of this version"
    assert doc["oof_sha256"] == locked_sha["oof_predictions.parquet"], f"ABORT: {label} Platt fit on a different OOF file"
    spec = read_json(files["models/model_spec.json"])
    R = {"label": label, "path": reg_path, "reg": reg, "version": str(reg["version"]), "reg_sha": sha256_file(reg_path),
         "files": files, "locked_sha": locked_sha, "cal_ref": cal_ref, "platt_path": cal_ref["path"], "platt_sha": platt_sha,
         "platt": {m: doc["params"][m] for m in MODELS_USED},
         "A_FEATS": reg["features"]["A"], "POOL": reg["features"]["BC_pool"], "BC_COLS": reg["features"]["BC_columns"],
         "WOE_KW": spec["A"]["woe_transform"],
         "A_BP": pickle.load(open(files["models/A/binning_process.pkl"], "rb")),
         "A_LR": pickle.load(open(files["models/A/lr_sklearn.pkl"], "rb")),
         "C_BST": lgb.Booster(model_file=files["models/C/lgbm.txt"])}
    assert R["C_BST"].num_trees() == reg["lightgbm_rounds"]["final_rounds"], f"ABORT: {label} LightGBM trees != locked final rounds"
    R["FEATS"] = list(dict.fromkeys(R["A_FEATS"] + R["POOL"]))
    return R

NF = load_registry(REG_PATH, "no-FTS")
assert NF["version"] == VERSION, f"ABORT: registry version {NF['version']} != config.registry.version {VERSION}"
REG, FILES, REG_SHA, CAL_REF = NF["reg"], NF["files"], NF["reg_sha"], NF["cal_ref"]
PLATT_PATH, PLATT_SHA, PLATT = NF["platt_path"], NF["platt_sha"], NF["platt"]
assert not [f for f in NF["FEATS"] if f.lower().startswith("fts_")], "ABORT: the no-FTS models use an fts_ feature"
FTSR = load_registry(FTS_REG_PATH, "FTS") if FTS_ON else None
if FTS_ON:
    assert os.path.abspath(os.path.dirname(FTS_REG_PATH)) != os.path.abspath(ARTIFACTS), "FTS registry must not be this variant's registry"

FDICT_PATH = os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")
FDICT, FDICT_SHA = pd.read_csv(FDICT_PATH), sha256_file(FDICT_PATH)
MODEL_ROOT = os.path.dirname(REG_PATH)

def tree_snapshot(root):
    """(size, mtime) of every file under a model folder, excluding this module's output folder and the open-questions file."""
    snap, out_abs, oq_abs = {}, os.path.abspath(OUT), os.path.abspath(OQ_PATH)
    for dp, _, fn in os.walk(root):
        a = os.path.abspath(dp)
        if a == out_abs or a.startswith(out_abs + os.sep):
            continue
        for f in fn:
            p = os.path.join(dp, f)
            if os.path.abspath(p) == oq_abs:
                continue
            st = os.stat(p)
            snap[p] = (st.st_size, st.st_mtime)
    return snap

def table_version(t):
    return int(spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"])

PROTECTED_TABLES = {}
_prot = [MASTER_TBL, f"{cat}.{sch}.acard_feature_log", f"{cat}.{sch}.{cfg['tables']['feature_log']}",
         f"{cat}.{sch}.{cfg['main_tables']['feature_log']}", E13_STORED, E13_STORED_FTS,
         f"{cat}.{sch}.{cfg['tables']['fwd_predictions']}", f"{cat}.{sch}.{cfg['tables']['stress_predictions']}"] + \
        [b["table"] for b in BENCHES if b.get("table")]
for t in dict.fromkeys(_prot):
    if spark.catalog.tableExists(t):
        PROTECTED_TABLES[t] = table_version(t)
SNAP0 = tree_snapshot(MODEL_ROOT)
SNAP0_FTS = tree_snapshot(FTS_ROOT) if FTS_ON else {}
for R in [NF] + ([FTSR] if FTS_ON else []):
    rg = R["reg"]
    print(f"{R['label']}: v{rg['version']} locked {rg['locked_at']} | signed {rg['signoff']['decision_rules_signed_by']} {rg['signoff']['signed_date']} | "
          f"{len(R['files'])} locked hashes OK | Platt OK ({R['platt_sha'][:12]}) | A {len(R['A_FEATS'])} features | C pool {len(R['POOL'])}, {R['C_BST'].num_trees()} trees")
    for m in MODELS_USED:
        p = R["platt"][m]
        print(f"   Platt {m}: a={p['a']:.4f} b={p['b']:.4f} (n={p['n']}, bads={p['n_bad']})")
print(f"Snapshots: {len(SNAP0)} files under {MODEL_ROOT}, {len(SNAP0_FTS)} under {FTS_ROOT} | protected tables {PROTECTED_TABLES}")

In [0]:
m_cols = spark.table(MASTER_TBL).columns
_lc = {c.lower(): c for c in m_cols}
WATCH_COLS = [w for w in WATCH if w in m_cols]
SEG_AVAILABLE = [c for c in SEGMENTS if c in m_cols]
SEG_MISSING = [c for c in SEGMENTS if c not in m_cols]
FTS_FLAG_COL = _lc.get(FTS_FLAG_NAME.lower())
assert FTS_FLAG_COL, f"FTS flag column {FTS_FLAG_NAME} not found (case-insensitive) in {MASTER_TBL}"
for b in BENCHES:                                    # benchmark columns read from acard_master (entries without a table)
    if not b.get("table"):
        b["_master_col"] = _lc.get(str(b["column"]).lower())
        assert b["_master_col"], f"benchmark column {b['column']} not found (case-insensitive) in {MASTER_TBL}"
ALL_FEATS = list(dict.fromkeys(NF["FEATS"] + (FTSR["FEATS"] if FTS_ON else [])))
assert not [f for f in ALL_FEATS if f not in m_cols], "model features missing in master"
keep = list(dict.fromkeys([id_col, "CREDITCARDNO", DATE_A, DECISION_A, "window", "sample_status", FTS_FLAG_COL] + WATCH_COLS + SEG_AVAILABLE
                          + [b["_master_col"] for b in BENCHES if b.get("_master_col")] + ALL_FEATS))
master = spark.table(MASTER_TBL).select(*keep).toPandas()
STATUS_COUNTS = master.groupby(["window", "sample_status"]).size().unstack(fill_value=0).reset_index()

# app_id (benchmark join key), same rule as s00_build_a_score_snapshot: CRNNO (as stored), or CREDITCARDNO where the
# normalised CRNNO (string, trim, strip '.0', strip leading zeros) is repeated in acard_master
def _nk(s):
    o = s.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")
    return o.mask(o == "", pd.NA)
_crn = _nk(master[id_col])
_dup = (_crn.notna() & _crn.duplicated(keep=False)).to_numpy()
_card = master["CREDITCARDNO"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
master["app_id"] = master[id_col].astype("string").str.strip().where(~_dup, _card).astype(str)
master["app_id_source"] = np.where(_dup, "CREDITCARDNO (CRNNO repeated)", "CRNNO")
assert master["app_id"].notna().all() and master["app_id"].is_unique, "app_id rule does not give one row per application"
_f = master[FTS_FLAG_COL].astype("string").str.strip()
master["has_fts_data"] = np.where(_f.notna() & (_f != ""), "Y", "N")          # segment only, never a feature

def scope(name):
    """One window via common.load_window (OOT read lock); approval base = approval_base_statuses (EXCLUDED left out)."""
    df = common.load_window(master, name, cfg, registry_path=REG_PATH)
    df = df[df["sample_status"].isin(BASE_STATUSES)].reset_index(drop=True)
    d, a = pd.to_datetime(df[DATE_A]), pd.to_datetime(df[DECISION_A])
    df["app_month"] = d.dt.to_period("M").astype(str)
    df["approve_month"] = a.dt.to_period("M").astype(str)
    df["gap_days"] = (a - d).dt.days
    return df

OOT, FWD, STRESS = scope("OOT"), scope("FWD"), scope("STRESS")
_d = pd.to_datetime(OOT[DATE_A])
OOT["oot_split"] = None
for q, (s, e) in SPLITS.items():
    OOT.loc[(_d >= s) & (_d < e), "oot_split"] = q
assert OOT["oot_split"].notna().all(), "OOT rows outside config.oot_report_splits"

def score_e13(df, R, prefix=""):
    """E13 scoring (s05_oot 'Score with the locked models'): raw PD -> Platt (DEV OOF) -> common score scaling."""
    XA = df[R["A_FEATS"]].apply(pd.to_numeric, errors="coerce")
    raw = {"A": R["A_LR"].predict_proba(np.asarray(R["A_BP"].transform(XA, metric="woe", **R["WOE_KW"]), dtype=float))[:, 1]}
    XR = common.prepare_raw(df, R["POOL"], FDICT)
    assert list(XR.columns) == R["BC_COLS"], f"{R['label']}: prepare_raw columns differ from the locked B/C columns"
    raw["C"] = R["C_BST"].predict(XR)
    for m in MODELS_USED:
        cal = common.apply_platt(raw[m], R["platt"][m])
        assert np.isfinite(cal).all() and ((cal > 0) & (cal < 1)).all(), f"{R['label']} {m}: PD outside (0, 1)"
        df[f"raw_{prefix}{m}"], df[f"pd_{prefix}{m}"], df[f"score_{prefix}{m}"] = raw[m], cal, common.pd_to_score(cal, cfg)
    return df

FTS_KEYS = [f"FTS_{m}" for m in MODELS_USED] if FTS_ON else []
SCORED_WINDOWS = []
for nm, df in (("OOT", OOT), ("FWD", FWD), ("STRESS", STRESS)):
    score_e13(df, NF)
    if FTS_ON:
        score_e13(df, FTSR, prefix="FTS_")
    SCORED_WINDOWS.append(nm)
assert "DEV" not in SCORED_WINDOWS                      # never score DEV in-sample

def spot_check(R, prefix, stored):
    """OOT scores reproduce the stored E13 scores of the same registry (1,000 rows, keyed on CREDITCARDNO)."""
    st = spark.table(stored).select("CREDITCARDNO", *[f"prob_{m}_cal" for m in MODELS_USED], *[f"score_{m}" for m in MODELS_USED]).toPandas()
    st["CREDITCARDNO"] = st["CREDITCARDNO"].astype(str)
    st = st[~st["CREDITCARDNO"].duplicated(keep=False)].set_index("CREDITCARDNO")
    card = OOT["CREDITCARDNO"].astype(str)
    cand = sorted(set(card[~card.duplicated(keep=False)]) & set(st.index))
    spot = set(np.random.default_rng(SEED).choice(cand, size=min(1000, len(cand)), replace=False))
    sel = np.where(card.isin(spot).to_numpy())[0]
    out = []
    for m in MODELS_USED:
        s_ = st.loc[card.iloc[sel]]
        out.append({"registry": R["label"], "version": R["version"], "model": m, "stored_table": stored, "n": len(sel),
                    "max_abs_pd_diff": float(np.abs(OOT[f"pd_{prefix}{m}"].to_numpy()[sel] - s_[f"prob_{m}_cal"].to_numpy()).max()),
                    "n_score_diff": int((OOT[f"score_{prefix}{m}"].to_numpy()[sel] != s_[f"score_{m}"].to_numpy()).sum())})
    return out

SPOT = pd.DataFrame(spot_check(NF, "", E13_STORED) + (spot_check(FTSR, "FTS_", E13_STORED_FTS) if FTS_ON else []))

# DEV second view: each no-FTS model's OOF predictions (never in-sample DEV scores), Platt-calibrated and scaled
OOF_RAW = pd.read_parquet(FILES["oof_predictions.parquet"])
OOF_RAW["app_id"] = OOF_RAW["app_id"].astype(str)
dev = common.load_window(master, "DEV", cfg, registry_path=REG_PATH)
dev_gb = dev[dev["sample_status"].isin(LABELLED)][[id_col, "sample_status", DATE_A] + WATCH_COLS].copy()
dev_gb[id_col] = dev_gb[id_col].astype(str)
DEVO = {}
for m in MODELS_USED:
    o = OOF_RAW[OOF_RAW["model"] == m][["app_id", "fold", "p_oof"]]
    assert o["app_id"].is_unique and o["p_oof"].notna().all(), f"OOF {m}: duplicate ids or missing predictions"
    j = o.merge(dev_gb, left_on="app_id", right_on=id_col, how="left", validate="1:1")
    assert j["sample_status"].notna().all(), f"OOF {m}: rows that are not DEV GOOD/BAD in master"
    j[f"pd_{m}"] = common.apply_platt(j["p_oof"].to_numpy(), PLATT[m])
    j[f"score_{m}"] = common.pd_to_score(j[f"pd_{m}"].to_numpy(), cfg)
    j["pd_source"] = "oof_predictions.parquet (p_oof) + Platt"
    DEVO[m] = j.reset_index(drop=True)

OOT_COUNTS = OOT["sample_status"].value_counts().reindex(BASE_STATUSES, fill_value=0)
print("OOT base:", len(OOT), OOT_COUNTS.to_dict(), "| by split:", OOT["oot_split"].value_counts().to_dict(),
      "| has_fts_data:", OOT["has_fts_data"].value_counts().to_dict())
print("FWD base:", len(FWD), FWD["sample_status"].value_counts().to_dict(), "| STRESS base:", len(STRESS), STRESS["sample_status"].value_counts().to_dict())
print("DEV OOF:", {m: (len(d), int((d['sample_status'] == 'BAD').sum())) for m, d in DEVO.items()}, "| DEV GOOD+BAD in master:", len(dev_gb))
print("Segments available:", SEG_AVAILABLE, "| missing:", SEG_MISSING, "| watch:", WATCH_COLS, "| FTS flag:", FTS_FLAG_COL)
display(SPOT)

In [0]:
# Table entries: read-only from the benchmark table, joined on app_id, one row per app_id asserted (never the source table,
# never load_a_score). Entries without a table: column of acard_master. Column names resolved case-insensitively;
# score oriented so that higher = lower risk (approve if score >= cut-off). Comparisons only where the benchmark is non-missing.
BENCH_COV, BENCH_TIMING, BENCH_DIR, BENCH_DIR_SUMM, BENCH_SRC, BENCH_UNIQUE = [], [], [], [], [], {}
for b in BENCHES:
    k, nm = b["key"], b["name"]
    sign = 1.0 if b["higher_is_lower_risk"] else -1.0
    if b.get("table"):
        bt = spark.table(b["table"])
        cmap = {c.lower(): c for c in bt.columns}
        col, jk = cmap.get(str(b["column"]).lower()), cmap.get(str(b["join_key"]).lower())
        dcol = cmap.get(str(b.get("date_column", "score_date")).lower())
        assert col and jk, f"{nm}: column {b['column']} / join key {b['join_key']} not in {b['table']}"
        bpd = bt.select(jk, col, *([dcol] if dcol else [])).toPandas()
        bpd[jk] = bpd[jk].astype(str).str.strip()
        BENCH_UNIQUE[k] = bool(bpd[jk].is_unique)
        assert BENCH_UNIQUE[k], f"{b['table']}: more than one row per {jk}"
        bpd = bpd.rename(columns={jk: "app_id", col: "_bscore", **({dcol: "_bdate"} if dcol else {})})
        offsets = ASCORE_OFFSETS if dcol else None
        src = {"benchmark": nm, "description": b["desc"], "table": b["table"], "source_table": b.get("source_table"), "column": col,
               "join_key": jk, "date_column": dcol, "score_month_offsets_used": offsets, "definition": b.get("definition"),
               "higher_is_lower_risk": b["higher_is_lower_risk"], "rows_in_table": len(bpd), "one_row_per_join_key": BENCH_UNIQUE[k]}
    else:
        offsets = None
        src = {"benchmark": nm, "description": b["desc"], "table": MASTER_TBL, "source_table": b.get("source_table"), "column": b["_master_col"],
               "join_key": "(same row)", "date_column": None, "score_month_offsets_used": None, "definition": b.get("definition"),
               "higher_is_lower_risk": b["higher_is_lower_risk"], "rows_in_table": None, "one_row_per_join_key": None}
    BENCH_SRC.append(src)
    for w, df in (("OOT", OOT), ("FWD", FWD), ("STRESS", STRESS)):
        if b.get("table"):
            mg = df[["app_id"]].merge(bpd, on="app_id", how="left", validate="m:1")
            assert len(mg) == len(df) and (mg["app_id"].to_numpy() == df["app_id"].to_numpy()).all(), f"{w}: benchmark merge changed rows"
            raw = pd.to_numeric(mg["_bscore"], errors="coerce")
            if offsets is not None:
                gap = (pd.to_datetime(mg["_bdate"]).dt.to_period("M") - pd.to_datetime(df[DATE_A]).dt.to_period("M")).map(
                    lambda x: x.n if pd.notna(x) else np.nan)
            else:
                gap = pd.Series(np.nan, index=df.index)
        else:
            raw = pd.to_numeric(df[b["_master_col"]], errors="coerce")
            gap = pd.Series(np.nan, index=df.index)
        joined = raw.notna().to_numpy()
        use = joined & (gap.isin(offsets).to_numpy() if offsets is not None else True)
        df[f"score_{k}"] = np.where(use, sign * raw.to_numpy(), np.nan)
        df[f"has_{k}"], df[f"pd_{k}"], df[f"bench_month_offset_{k}"] = use, np.nan, gap.to_numpy()
        stt = df["sample_status"].to_numpy()
        gb, bd = np.isin(stt, LABELLED), stt == "BAD"
        BENCH_COV.append({"benchmark": nm, "window": w, "approval_base_n": len(df), "joined": int(joined.sum()),
                          "joined_but_outside_score_months": int((joined & ~use).sum()), "used": int(use.sum()),
                          "coverage_approval_base": use.mean(), "good_bad_n": int(gb.sum()), "good_bad_used": int((use & gb).sum()),
                          "coverage_good_bad": (use & gb).sum() / gb.sum() if gb.any() else np.nan,
                          "bads_n": int(bd.sum()), "bads_used": int((use & bd).sum()),
                          **{f"coverage_{s}": use[stt == s].mean() if (stt == s).any() else np.nan for s in BASE_STATUSES}})
        if offsets is not None:
            vc = gap[joined].value_counts().sort_index()
            BENCH_TIMING += [{"benchmark": nm, "window": w, "months_score_date_minus_application": int(kk), "n": int(v), "used": int(kk) in offsets}
                             for kk, v in vc.items()]
    # Direction check on OOT GOOD/BAD with the benchmark: bad rate by benchmark decile (1 = lowest oriented score = highest risk)
    d = OOT[OOT["sample_status"].isin(LABELLED) & OOT[f"has_{k}"]].copy()
    d["y"] = (d["sample_status"] == "BAD").astype(int)
    d["decile"] = pd.qcut(d[f"score_{k}"].rank(method="first"), 10, labels=False) + 1
    t = d.groupby("decile").agg(n=("y", "size"), bads=("y", "sum"), score_min=(f"score_{k}", "min"), score_max=(f"score_{k}", "max")).reset_index()
    t["bad_rate"] = t["bads"] / t["n"]
    BENCH_DIR.append(t.assign(benchmark=nm))
    auc = roc_auc_score(d["y"], -d[f"score_{k}"])
    rho = spearmanr(t["decile"], t["bad_rate"]).correlation
    BENCH_DIR_SUMM.append({"benchmark": nm, "n_good_bad": len(d), "bads": int(d["y"].sum()), "auc_oriented": auc, "gini": 2 * auc - 1,
                           "spearman_decile_vs_bad_rate": rho, "direction_ok (bad rate falls as score rises)": bool(rho < 0 and auc > 0.5)})
BENCH_COV, BENCH_TIMING, BENCH_SRC = pd.DataFrame(BENCH_COV), pd.DataFrame(BENCH_TIMING), pd.DataFrame(BENCH_SRC)
BENCH_DIR = pd.concat(BENCH_DIR, ignore_index=True) if BENCH_DIR else pd.DataFrame()
BENCH_DIR_SUMM = pd.DataFrame(BENCH_DIR_SUMM)
assert BENCH_DIR_SUMM.empty or BENCH_DIR_SUMM.iloc[:, -1].all(), "benchmark direction check failed: check higher_is_lower_risk"
display(BENCH_SRC)
display(BENCH_COV[BENCH_COV["window"] == "OOT"])
display(BENCH_DIR_SUMM)

In [0]:
def _counts(st, mask=None):
    st = st if mask is None else st[mask]
    return {s: int((st == s).sum()) for s in ["GOOD", "BAD"] + OTHER}

def boot_idx(bad_gb, key):
    """Stratified (GOOD/BAD) bootstrap indices over the GOOD+BAD rows; seed = config.selection.bootstrap.seed + fixed key."""
    rng = np.random.default_rng([BOOT["seed"], key])
    ib, ig = np.where(bad_gb)[0], np.where(~bad_gb)[0]
    if not BOOT.get("stratified", True):
        return rng.choice(len(bad_gb), (BOOT["n"], len(bad_gb))).astype(np.int32)
    return np.concatenate([rng.choice(ib, (BOOT["n"], len(ib))), rng.choice(ig, (BOOT["n"], len(ig)))], axis=1).astype(np.int32)

def band_lower(score):
    return (np.floor(np.asarray(score, dtype=float) / BW) * BW).astype(int)

def band_table(F, m):
    """Score bands of BW points, best (highest score) first; approved if score >= band lower bound."""
    sc, pd_, st = F[f"score_{m}"].to_numpy(), F[f"pd_{m}"].to_numpy(), F["sample_status"].to_numpy()
    lb, gb, bad = band_lower(sc), np.isin(st, LABELLED), st == "BAD"
    N, TB, TGB = len(F), int(bad.sum()), int(gb.sum())
    rows = []
    for b in sorted(np.unique(lb), reverse=True):
        mk = lb == b
        c = _counts(st, mk)
        ngb = c["GOOD"] + c["BAD"]
        rows.append({"band": f"{b}-{b + BW - 1}", "band_lower": b, "n": int(mk.sum()), "share": mk.sum() / N, **c,
                     "n_good_bad": ngb, "bad_rate": c["BAD"] / ngb if ngb else np.nan,
                     "good_bad_odds": c["GOOD"] / c["BAD"] if c["BAD"] else np.nan,
                     "mean_pd_good_bad": pd_[mk & gb].mean() if (mk & gb).any() else np.nan})
    t = pd.DataFrame(rows)
    t["pd_minus_bad_rate"] = t["mean_pd_good_bad"] - t["bad_rate"]
    t["cum_n_approved"] = t["n"].cumsum()
    t["cum_approval_rate"] = t["cum_n_approved"] / N
    t["cum_good_bad_approved"] = t["n_good_bad"].cumsum()
    t["cum_approval_rate_good_bad_only"] = t["cum_good_bad_approved"] / TGB
    t["cum_bads_approved"] = t["BAD"].cumsum()
    t["cum_approved_bad_rate"] = t["cum_bads_approved"] / t["cum_good_bad_approved"].replace(0, np.nan)
    t["cum_bads_declined"] = TB - t["cum_bads_approved"]
    t["cum_share_of_bads_declined"] = t["cum_bads_declined"] / TB
    for s in OTHER:
        t[f"cum_{s}_approved"] = t[s].cumsum()
    prev = t["bad_rate"].ffill().shift(1)            # bad rate of the nearest better band with GOOD/BAD rows
    t["non_monotonic"] = t["bad_rate"].notna() & prev.notna() & (t["bad_rate"] < prev)
    return t

BANDS = {m: band_table(OOT, m) for m in MODELS_USED}
for m, t in BANDS.items():
    assert t["n"].sum() == len(OOT) and t["BAD"].sum() == OOT_COUNTS["BAD"] and t["GOOD"].sum() == OOT_COUNTS["GOOD"]
    print(f"{m}: {len(t)} bands {t['band_lower'].min()}..{t['band_lower'].max()} | non-monotonic bands: {int(t['non_monotonic'].sum())} "
          f"| bands with < 30 GOOD+BAD: {int((t['n_good_bad'] < 30).sum())}")
display(BANDS["A"][["band", "n", "share", "GOOD", "BAD", "INDETERMINATE", "IMMATURE", "EXCLUDED_PERF", "bad_rate", "mean_pd_good_bad", "cum_approval_rate", "cum_approved_bad_rate", "non_monotonic"]])

In [0]:
def cutoff_for_rate(score, r):
    """Highest integer cut-off with approval rate (score >= cut-off) >= r."""
    s = np.sort(np.asarray(score))[::-1]
    return int(s[max(int(np.ceil(r * len(s))) - 1, 0)])

def grid_table(F, m, I, cuts=None, others_available=True, view="OOT"):
    """One row per grid approval rate. cuts: {rate: cut-off} to apply fixed cut-offs (else derived on F).
    I: bootstrap indices over F's GOOD+BAD rows (in row order)."""
    sc, pd_, st = F[f"score_{m}"].to_numpy(), F[f"pd_{m}"].to_numpy(), F["sample_status"].to_numpy()
    gb, bad = np.isin(st, LABELLED), st == "BAD"
    bad_gb = bad[gb]
    N, TB, TGB = len(F), int(bad.sum()), int(gb.sum())
    rows = []
    for r in GRID:
        c = cuts[r] if cuts is not None else cutoff_for_rate(sc, r)
        a = sc >= c
        a_gb = a[gb]
        n_app, gb_app, b_app = int(a.sum()), int(a_gb.sum()), int((a & bad).sum())
        row = {"view": view, "model": m, "approval_rate_target": r, "cutoff_score": c,
               "n_tied_at_cutoff": int((sc == c).sum()), "approval_rate_achieved": n_app / N,
               "approval_rate_if_cutoff_plus_1": float((sc > c).mean()),
               "approval_rate_good_bad_only": gb_app / TGB,
               "n_total": N, "n_approved": n_app, "n_declined": N - n_app,
               "good_bad_total": TGB, "good_bad_approved": gb_app, "good_bad_declined": TGB - gb_app,
               "bads_total": TB, "bads_approved": b_app, "bads_declined": TB - b_app,
               "share_of_bads_declined": (TB - b_app) / TB,
               "approved_bad_rate": b_app / gb_app if gb_app else np.nan}
        bb = (bad_gb & a_gb)[I].sum(axis=1)
        nb = a_gb[I].sum(axis=1)
        br = bb / np.where(nb == 0, np.nan, nb)
        row["approved_bad_rate_ci_low"], row["approved_bad_rate_ci_high"] = np.nanpercentile(br, 2.5), np.nanpercentile(br, 97.5)
        row["declined_bad_rate"] = (TB - b_app) / (TGB - gb_app) if TGB - gb_app else np.nan
        row["expected_bad_rate_approved"] = pd_[a & gb].mean() if (a & gb).any() else np.nan
        row["expected_minus_actual"] = row["expected_bad_rate_approved"] - row["approved_bad_rate"]
        for s in OTHER:
            if others_available:
                ns = st == s
                row[f"{s}_approved"], row[f"{s}_declined"] = int((ns & a).sum()), int((ns & ~a).sum())
                row[f"{s}_share_among_approved"] = (ns & a).sum() / n_app if n_app else np.nan
            else:
                row[f"{s}_approved"] = row[f"{s}_declined"] = row[f"{s}_share_among_approved"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

GB_OOT = OOT["sample_status"].isin(LABELLED).to_numpy()
I_OOT = boot_idx((OOT["sample_status"] == "BAD").to_numpy()[GB_OOT], 1)
GRID_OOT = {m: grid_table(OOT, m, I_OOT, view="OOT") for m in MODELS_USED}
CUTS = {m: dict(zip(GRID_OOT[m]["approval_rate_target"], GRID_OOT[m]["cutoff_score"])) for m in MODELS_USED}

# DEV second view: OOF predictions exist only for DEV GOOD + BAD, so the base there is GOOD + BAD
# (INDETERMINATE / IMMATURE / EXCLUDED_PERF have no OOF prediction and in-sample scores are not allowed).
GRID_DEV = {}
for k, m in enumerate(MODELS_USED):
    F = DEVO[m]
    I_dev = boot_idx((F["sample_status"] == "BAD").to_numpy(), 10 + k)
    GRID_DEV[m] = grid_table(F, m, I_dev, others_available=False, view="DEV_OOF (base = GOOD+BAD)")

for m in MODELS_USED:
    print(m, "OOT cut-offs:", CUTS[m])
cols = ["model", "approval_rate_target", "cutoff_score", "n_tied_at_cutoff", "approval_rate_achieved", "approval_rate_good_bad_only",
        "approved_bad_rate", "approved_bad_rate_ci_low", "approved_bad_rate_ci_high", "declined_bad_rate", "share_of_bads_declined",
        "expected_bad_rate_approved", "INDETERMINATE_share_among_approved", "IMMATURE_share_among_approved"]
display(pd.concat([GRID_OOT[m][cols] for m in MODELS_USED]))

In [0]:
def compare_models(F, x, y, I, cuts_x, cuts_y, label="OOT"):
    """x vs y at the same approval rate, each with its own cut-off. Net bads avoided by x = bads declined by x - by y.
    Paired bootstrap: the same stratified GOOD/BAD resamples for both scores."""
    sx, sy, st = F[f"score_{x}"].to_numpy(), F[f"score_{y}"].to_numpy(), F["sample_status"].to_numpy()
    gb, bad = np.isin(st, LABELLED), st == "BAD"
    bad_gb = bad[gb].astype(int)
    rows, swaps = [], []
    for r in GRID:
        ax, ay = sx >= cuts_x[r], sy >= cuts_y[r]
        dvec = bad_gb * ((~ax[gb]).astype(int) - (~ay[gb]).astype(int))      # +1 = bad declined by x only
        boot = dvec[I].sum(axis=1)
        bx, by = int((bad & ~ax).sum()), int((bad & ~ay).sum())
        rows.append({"split": label, "pair": f"{x} vs {y}", "n": len(F), "approval_rate_target": r, f"cutoff_{x}": cuts_x[r], f"cutoff_{y}": cuts_y[r],
                     f"approval_rate_{x}": ax.mean(), f"approval_rate_{y}": ay.mean(),
                     "bads_total": int(bad.sum()), f"bads_declined_{x}": bx, f"bads_declined_{y}": by,
                     f"net_bads_avoided_by_{x}": bx - by, "net_ci_low": float(np.percentile(boot, 2.5)),
                     "net_ci_high": float(np.percentile(boot, 97.5)), "p_delta_le_0": float((boot <= 0).mean()),
                     f"approved_bad_rate_{x}": (bad & ax).sum() / (gb & ax).sum(), f"approved_bad_rate_{y}": (bad & ay).sum() / (gb & ay).sum()})
        for g, mk in {f"declined by {x} only": ~ax & ay, f"declined by {y} only": ax & ~ay,
                      "approved by both": ax & ay, "declined by both": ~ax & ~ay}.items():
            c = _counts(st, mk)
            ngb = c["GOOD"] + c["BAD"]
            swaps.append({"split": label, "pair": f"{x} vs {y}", "approval_rate_target": r, "group": g, "n": int(mk.sum()), **c,
                          "n_good_bad": ngb, "bad_rate": c["BAD"] / ngb if ngb else np.nan})
    return pd.DataFrame(rows), pd.DataFrame(swaps)

def agreement(F, label="OOT", x="A", y="C"):
    """Spearman of the two scores and a decile cross-table (rank deciles, 1 = lowest risk: calibrated PD for the
    models, -score for a benchmark without PD)."""
    def risk(m):
        return F[f"pd_{m}"] if F[f"pd_{m}"].notna().all() else -F[f"score_{m}"]
    dec = {m: pd.qcut(risk(m).rank(method="first"), 10, labels=range(1, 11)).astype(int).to_numpy() for m in (x, y)}
    diff = np.abs(dec[x] - dec[y])
    gb = F["sample_status"].isin(LABELLED).to_numpy()
    summ = {"split": label, "pair": f"{x} vs {y}", "n": len(F),
            "spearman_score_all": float(spearmanr(F[f"score_{x}"], F[f"score_{y}"]).correlation),
            "spearman_score_good_bad": float(spearmanr(F.loc[gb, f"score_{x}"], F.loc[gb, f"score_{y}"]).correlation),
            "decile_diagonal_share": float((diff == 0).mean()), "decile_diagonal_adjacent_share": float((diff <= 1).mean())}
    ct = pd.crosstab(pd.Series(dec[x], name=f"decile_{x} (rows)"), pd.Series(dec[y], name=f"decile_{y}")).reset_index()
    return summ, ct

# 3.1 C vs A (no-FTS), OOT approval base
C_VS_A, SWAP = compare_models(OOT, "C", "A", I_OOT, CUTS["C"], CUTS["A"])
for (r, g), t in SWAP.groupby(["approval_rate_target", "split"]):
    assert t["n"].sum() == len(OOT), "swap groups do not reconcile"
AG = [pd.DataFrame([agreement(OOT)[0]])]
AG_CT = [("Decile cross-table A (rows) vs C, OOT", agreement(OOT)[1])]

# 3.2 Each no-FTS model vs each benchmark, on the OOT applicants with a non-missing benchmark (every score with its own
# cut-off at the same approval rate on that population); coverage next to every result
BR = {}
for j, k in enumerate(BENCH_KEYS):
    nm = BENCH_NAME_OF[k]
    FB = OOT[OOT[f"has_{k}"]].reset_index(drop=True)
    gbb = FB["sample_status"].isin(LABELLED).to_numpy()
    IB = boot_idx((FB["sample_status"] == "BAD").to_numpy()[gbb], 2 + j)
    cov = {"benchmark": nm, "comparison_n": len(FB), "coverage_approval_base": len(FB) / len(OOT),
           "coverage_good_bad": gbb.sum() / OOT["sample_status"].isin(LABELLED).sum(), "bads_covered": int((FB["sample_status"] == "BAD").sum())}
    allm = MODELS_USED + [k]
    G = {m: grid_table(FB, m, IB, view=f"OOT with {nm} (n={len(FB)}, {len(FB) / len(OOT):.1%} of base)") for m in allm}
    CU = {m: dict(zip(G[m]["approval_rate_target"], G[m]["cutoff_score"])) for m in allm}
    VS, SW, AGB = {}, [], []
    for x in MODELS_USED:
        c, s = compare_models(FB, x, k, IB, CU[x], CU[k], label=f"OOT with {nm}")
        for i, (ck, cv) in enumerate(cov.items()):
            c.insert(i, ck, cv)
        VS[x] = c
        SW.append(s.assign(benchmark=nm))
        a, ct = agreement(FB, label=f"OOT with {nm}", x=x, y=k)
        AGB.append((a, ct))
    SW = pd.concat(SW, ignore_index=True)
    for (p, r), t in SW.groupby(["pair", "approval_rate_target"]):
        assert t["n"].sum() == len(FB), f"{nm}: swap groups do not reconcile"
    BR[k] = {"name": nm, "F": FB, "I": IB, "cov": cov, "GRID": G, "CUTS": CU, "BANDS": band_table(FB, k), "VS": VS, "SWAP": SW, "AG": AGB}
    AG += [pd.DataFrame([a]) for a, _ in AGB]
    AG_CT += [(f"Decile cross-table {a['pair']}, OOT with {nm}", ct) for a, ct in AGB]
VS_BENCH = pd.concat([BR[k]["VS"][x] for k in BENCH_KEYS for x in MODELS_USED], ignore_index=True) if BR else pd.DataFrame()

# 3.3 FTS models (read-only registry, E13 scoring) vs no-FTS: net bads avoided by the FTS model, same OOT applicants,
# paired bootstrap on the same resamples as C vs A; split by has_fts_data as a segment (full-OOT cut-offs)
GRID_FTS, CUTS_FTS, VS_FTS, FTS_SWAP, FTS_SEG, FTS_SEG_SWAP = {}, {}, {}, [], [], []
if FTS_ON:
    for m in MODELS_USED:
        f = f"FTS_{m}"
        GRID_FTS[m] = grid_table(OOT, f, I_OOT, view=f"OOT, FTS v{FTSR['version']} model {m}")
        CUTS_FTS[m] = dict(zip(GRID_FTS[m]["approval_rate_target"], GRID_FTS[m]["cutoff_score"]))
        c, s = compare_models(OOT, f, m, I_OOT, CUTS_FTS[m], CUTS[m], label="OOT")
        VS_FTS[m] = c
        FTS_SWAP.append(s)
        a, ct = agreement(OOT, label="OOT", x=m, y=f)
        AG.append(pd.DataFrame([a]))
        AG_CT.append((f"Decile cross-table {m} no-FTS (rows) vs FTS {m}, OOT", ct))
    for kk, val in enumerate(["Y", "N"]):
        Fs = OOT[OOT["has_fts_data"] == val].reset_index(drop=True)
        gbs = Fs["sample_status"].isin(LABELLED).to_numpy()
        Is = boot_idx((Fs["sample_status"] == "BAD").to_numpy()[gbs], 40 + kk)
        for m in MODELS_USED:
            c, s = compare_models(Fs, f"FTS_{m}", m, Is, CUTS_FTS[m], CUTS[m], label=f"has_fts_data={val} (full-OOT cut-offs)")
            FTS_SEG.append(c)
            FTS_SEG_SWAP.append(s)
    FTS_SWAP, FTS_SEG, FTS_SEG_SWAP = pd.concat(FTS_SWAP, ignore_index=True), pd.concat(FTS_SEG, ignore_index=True), pd.concat(FTS_SEG_SWAP, ignore_index=True)
    for (p, r), t in FTS_SWAP.groupby(["pair", "approval_rate_target"]):
        assert t["n"].sum() == len(OOT), "FTS swap groups do not reconcile"
AG = pd.concat(AG, ignore_index=True)

display(C_VS_A)
if BR:
    display(VS_BENCH[["benchmark", "pair", "comparison_n", "coverage_approval_base", "approval_rate_target", "net_ci_low", "net_ci_high", "p_delta_le_0"]])
if FTS_ON:
    display(pd.concat(VS_FTS.values())[["pair", "approval_rate_target", "net_ci_low", "net_ci_high", "p_delta_le_0"]])
display(AG)

In [0]:
QB, QG, QC, QS, QA, QCT = [], [], [], [], [], []
QFTS = []
for k, q in enumerate(SPLITS):
    Fq = OOT[OOT["oot_split"] == q].reset_index(drop=True)
    gbq = Fq["sample_status"].isin(LABELLED).to_numpy()
    Iq = boot_idx((Fq["sample_status"] == "BAD").to_numpy()[gbq], 20 + k)
    for m in MODELS_USED:
        QB.append(band_table(Fq, m).assign(split=q, model=m))
        QG.append(grid_table(Fq, m, Iq, cuts=CUTS[m], view=f"OOT {q} (full-OOT cut-offs)"))
    c, s = compare_models(Fq, "C", "A", Iq, CUTS["C"], CUTS["A"], label=q)
    QC.append(c); QS.append(s)
    a, ct = agreement(Fq, label=q)
    QA.append(a); QCT.append((f"Decile cross-table A (rows) vs C, {q}", ct))
    if FTS_ON:                                   # FTS vs no-FTS by quarter, same resamples, full-OOT cut-offs
        for m in MODELS_USED:
            c, s = compare_models(Fq, f"FTS_{m}", m, Iq, CUTS_FTS[m], CUTS[m], label=q)
            a, _ = agreement(Fq, label=q, x=m, y=f"FTS_{m}")
            QFTS += [(f"FTS {m} vs no-FTS {m}, {q}", c), (f"Swap sets FTS {m} vs no-FTS {m}, {q}", s), (f"Agreement no-FTS {m} vs FTS {m}, {q}", pd.DataFrame([a]))]
QB, QG, QC, QS, QA = pd.concat(QB), pd.concat(QG), pd.concat(QC), pd.concat(QS), pd.DataFrame(QA)

# Benchmarks by quarter: same applicants with the benchmark, cut-offs fixed from the full covered OOT population
QVB = []
for j, (k, R) in enumerate(BR.items()):
    nm = R["name"]
    for kq, q in enumerate(SPLITS):
        Fq = R["F"][R["F"]["oot_split"] == q].reset_index(drop=True)
        gbq = Fq["sample_status"].isin(LABELLED).to_numpy()
        Iq = boot_idx((Fq["sample_status"] == "BAD").to_numpy()[gbq], 30 + 10 * j + kq)
        QVB.append((f"{nm} grid, {q} (n={len(Fq)})", grid_table(Fq, k, Iq, cuts=R["CUTS"][k], view=f"OOT {q} with {nm}")))
        for x in MODELS_USED:
            c, s = compare_models(Fq, x, k, Iq, R["CUTS"][x], R["CUTS"][k], label=q)
            a, _ = agreement(Fq, label=q, x=x, y=k)
            QVB += [(f"{x} vs {nm}, {q}", c), (f"Swap sets {x} vs {nm}, {q}", s), (f"Agreement {x} vs {nm}, {q}", pd.DataFrame([a]))]
display(QG[["view", "model", "approval_rate_target", "cutoff_score", "approval_rate_achieved", "approved_bad_rate",
            "approved_bad_rate_ci_low", "approved_bad_rate_ci_high", "share_of_bads_declined"]])
display(QC)

In [0]:
imm = (OOT["sample_status"] == "IMMATURE").to_numpy()
IMM_RATE = imm.mean()

def gap_stats(g):
    return pd.Series({"gap_days_mean": g.mean(), "gap_days_median": g.median(), "gap_days_p90": g.quantile(0.9), "gap_days_max": g.max()}) if len(g) else pd.Series(dtype=float)

rows = []
for mon, g in OOT.groupby("app_month"):
    i = g["sample_status"] == "IMMATURE"
    ad = pd.to_datetime(g.loc[i, DECISION_A])
    rows.append({"app_month": mon, "n": len(g), **_counts(g["sample_status"]), "immature_rate": i.mean(),
                 **gap_stats(g.loc[i, "gap_days"]).add_prefix("IMMATURE_").to_dict(),
                 **gap_stats(g.loc[~i, "gap_days"]).add_prefix("others_").to_dict(),
                 "IMMATURE_approvedate_min": ad.min(), "IMMATURE_approvedate_max": ad.max()})
IMM_MONTH = pd.DataFrame(rows)
IMM_APPROVE = (OOT.assign(imm=imm).groupby("approve_month").agg(n=("imm", "size"), IMMATURE=("imm", "sum"))
               .assign(immature_rate=lambda d: d["IMMATURE"] / d["n"]).reset_index())
IMM_GAP = pd.DataFrame([gap_stats(OOT.loc[imm, "gap_days"]).rename("IMMATURE"),
                        gap_stats(OOT.loc[~imm, "gap_days"]).rename("all other OOT")]).reset_index(names="group")

IMM_BAND, FLAGS = [], []
for m in MODELS_USED:
    lb = band_lower(OOT[f"score_{m}"])
    t = pd.DataFrame({"band_lower": lb, "imm": imm}).groupby("band_lower").agg(n=("imm", "size"), IMMATURE=("imm", "sum")).reset_index()
    t = t.sort_values("band_lower", ascending=False)
    t["band"] = t["band_lower"].map(lambda b: f"{b}-{b + BW - 1}")
    t["immature_rate"] = t["IMMATURE"] / t["n"]
    t["ratio_to_overall"] = t["immature_rate"] / IMM_RATE
    t["flag_concentrated"] = (t["n"] >= IMM_BAND_MIN_N) & (t["ratio_to_overall"] >= IMM_RATIO_FLAG)
    IMM_BAND.append(t.assign(model=m))
    tt = t[t["n"] >= IMM_BAND_MIN_N]
    p = chi2_contingency(np.c_[tt["IMMATURE"], tt["n"] - tt["IMMATURE"]])[1]
    FLAGS.append({"check": f"IMMATURE by {m} score band (bands with n >= {IMM_BAND_MIN_N}, chi-square)", "p_value": p,
                  "detail": f"bands with rate >= {IMM_RATIO_FLAG}x overall: {', '.join(t.loc[t['flag_concentrated'], 'band']) or 'none'}",
                  "flag": bool(p < IMM_P_FLAG and t["flag_concentrated"].any())})
IMM_BAND = pd.concat(IMM_BAND)
q = OOT.assign(imm=imm).groupby("oot_split").agg(n=("imm", "size"), IMMATURE=("imm", "sum"))
q["immature_rate"] = q["IMMATURE"] / q["n"]
q["share_of_all_IMMATURE"] = q["IMMATURE"] / imm.sum()
q["share_of_base"] = q["n"] / len(OOT)
p_q = chi2_contingency(np.c_[q["IMMATURE"], q["n"] - q["IMMATURE"]])[1]
r_q = q.loc["Q2", "immature_rate"] / q.loc["Q1", "immature_rate"] if q.loc["Q1", "immature_rate"] > 0 else np.inf
FLAGS.insert(0, {"check": "IMMATURE concentrated in Q2 (rate Q2 / Q1, chi-square)", "p_value": p_q,
                 "detail": f"rate Q1 {q.loc['Q1', 'immature_rate']:.4f}, Q2 {q.loc['Q2', 'immature_rate']:.4f}, ratio {r_q:.2f}; "
                           f"Q2 holds {q.loc['Q2', 'share_of_all_IMMATURE']:.1%} of IMMATURE vs {q.loc['Q2', 'share_of_base']:.1%} of the base",
                 "flag": bool(p_q < IMM_P_FLAG and r_q >= IMM_RATIO_FLAG)})
IMM_Q = q.reset_index()
IMM_FLAGS = pd.DataFrame(FLAGS)
display(IMM_FLAGS)
display(IMM_MONTH[["app_month", "n", "IMMATURE", "immature_rate", "IMMATURE_gap_days_median", "others_gap_days_median", "IMMATURE_approvedate_min", "IMMATURE_approvedate_max"]])

In [0]:
# Diagnosis only: no labels are used for FWD / STRESS. PSI on the BW-point score bands, OOT base as reference.
# Benchmarks: only applicants with the benchmark, cut-offs from the covered OOT population.
SCORE_ITEMS = [(m, m, None, CUTS[m]) for m in MODELS_USED] + [(k, BENCH_NAME_OF[k], f"has_{k}", BR[k]["CUTS"][k]) for k in BENCH_KEYS]
REC = []
for m, label, hcol, cuts_m in SCORE_ITEMS:
    sel = (lambda df, h=hcol: df[df[h]]) if hcol else (lambda df: df)
    O, Fw, S = sel(OOT), sel(FWD), sel(STRESS)
    ref = pd.Series(band_lower(O[f"score_{m}"]))
    groups = [("OOT", "ALL", O)] + [(w, "ALL", df) for w, df in (("FWD", Fw), ("STRESS", S)) if len(df)] + \
             [(w, mon, g) for w, df in (("FWD", Fw), ("STRESS", S)) for mon, g in df.groupby("app_month")]
    for w, mon, g in groups:
        s = g[f"score_{m}"].to_numpy()
        row = {"model": label, "window": w, "app_month": mon, "n": len(g), **_counts(g["sample_status"]),
               "mean_score": s.mean(), "median_score": float(np.median(s)),
               "score_psi_vs_oot": common.psi_from_bins(ref, pd.Series(band_lower(s)), EPS)}
        for r in GRID:                                   # cut-off of each score at rate r: see grid_oot_<model> / vs_benchmarks
            row[f"share_ge_cutoff_at_{r:g}"] = float((s >= cuts_m[r]).mean())
        REC.append(row)
REC = pd.DataFrame(REC)
display(REC[REC["app_month"] == "ALL"])

In [0]:
def watch_frame(df):
    out = {}
    for w in WATCH_COLS:
        if w.upper() == "AGE":
            b = pd.cut(pd.to_numeric(df[w], errors="coerce"), AGE_BANDS, right=False)
            out["AGE_BAND"] = b.astype(object).where(b.notna(), "MISSING").astype(str)
        else:
            v = _norm(df[w])
            v = v.astype(object).where(v.notna(), "MISSING").astype(str)
            top = v.value_counts().index[:TOP_N_WATCH]
            out[w] = v.where(v.isin(top), "OTHER")
    return pd.DataFrame(out, index=df.index)

WF = watch_frame(OOT)
st_all = OOT["sample_status"].to_numpy()
SEGROWS = []
key = 100
for dim in WF.columns:
    for val in sorted(WF[dim].unique()):
        for m, label, hcol, cuts_m in SCORE_ITEMS:
            # benchmark rows: applicants with the benchmark only, cut-offs from the covered OOT population
            cov = OOT[hcol].to_numpy() if hcol else np.ones(len(OOT), bool)
            mk = (WF[dim] == val).to_numpy() & cov
            if not mk.any():
                continue
            gbm, badm = np.isin(st_all, LABELLED) & mk, (st_all == "BAD") & mk
            n_bads = int(badm.sum())
            with_ci = n_bads >= SEG_MIN_BADS
            key += 1
            I_s = boot_idx(badm[gbm], key) if with_ci else None
            sc = OOT[f"score_{m}"].to_numpy()
            for r in GRID:
                a = sc >= cuts_m[r]
                gb_app, b_app = int((a & gbm).sum()), int((a & badm).sum())
                row = {"dim": dim, "value": val, "model": label, "approval_rate_target": r, "cutoff_score": cuts_m[r],
                       "n": int(mk.sum()), "n_approved": int((a & mk).sum()), "n_declined": int((~a & mk).sum()),
                       "approval_rate": (a & mk).sum() / mk.sum(), "approval_rate_good_bad_only": gb_app / gbm.sum() if gbm.any() else np.nan,
                       "good_bad": int(gbm.sum()), "bads": n_bads, "bads_approved": b_app, "bads_declined": n_bads - b_app,
                       "approved_bad_rate": b_app / gb_app if gb_app else np.nan,
                       "note": "" if with_ci else f"< {SEG_MIN_BADS} bads: bad rate and count only"}
                if with_ci:
                    ag = a[gbm]
                    br = (badm[gbm] & ag)[I_s].sum(axis=1) / np.where(ag[I_s].sum(axis=1) == 0, np.nan, ag[I_s].sum(axis=1))
                    row["approved_bad_rate_ci_low"], row["approved_bad_rate_ci_high"] = np.nanpercentile(br, 2.5), np.nanpercentile(br, 97.5)
                else:
                    row["approved_bad_rate_ci_low"] = row["approved_bad_rate_ci_high"] = np.nan
                SEGROWS.append(row)
SEGROWS = pd.DataFrame(SEGROWS)
SEG_AVAIL = pd.DataFrame([{"segment_column": c, "in_acard_master": c in m_cols} for c in SEGMENTS])
display(SEGROWS[(SEGROWS["approval_rate_target"] == 0.8) & (SEGROWS["model"] == "A")])

In [0]:
def pct_ci(v, lo, hi):
    return f"{v:.2%} [{lo:.2%}, {hi:.2%}]"

def net_ci(t, x, r):
    v = t.set_index("approval_rate_target").loc[r]
    return f"{int(v[f'net_bads_avoided_by_{x}'])} [{v['net_ci_low']:g}, {v['net_ci_high']:g}]", v["p_delta_le_0"]

SUMMARY = []
for r in GRID:
    row = {"approval_rate_target": r}
    for m in MODELS_USED:
        g = GRID_OOT[m].set_index("approval_rate_target").loc[r]
        row.update({f"{m}_cutoff_score": int(g["cutoff_score"]), f"{m}_approval_rate_achieved": g["approval_rate_achieved"],
                    f"{m}_approved_bad_rate": g["approved_bad_rate"],
                    f"{m}_approved_bad_rate_CI": pct_ci(g["approved_bad_rate"], g["approved_bad_rate_ci_low"], g["approved_bad_rate_ci_high"]),
                    f"{m}_share_of_bads_declined": g["share_of_bads_declined"],
                    f"{m}_INDETERMINATE_share": g["INDETERMINATE_share_among_approved"],
                    f"{m}_IMMATURE_share": g["IMMATURE_share_among_approved"],
                    f"{m}_EXCLUDED_PERF_share": g["EXCLUDED_PERF_share_among_approved"]})
    s, p = net_ci(C_VS_A, "C", r)
    row.update({"net_bads_avoided_C_vs_A_CI": s, "P(delta <= 0) C vs A": p})
    for k in BENCH_KEYS:        # on the OOT applicants with the benchmark; every score with its own cut-off on that population
        nm, R = BENCH_NAME_OF[k], BR[k]
        for x in MODELS_USED:
            s, p = net_ci(R["VS"][x], x, r)
            row.update({f"net_bads_avoided_{x}_vs_{nm}_CI": s, f"P(delta <= 0) {x} vs {nm}": p})
        row[f"{nm} comparison population"] = f"{len(R['F'])} of {len(OOT)} ({len(R['F']) / len(OOT):.1%})"
    if FTS_ON:
        for m in MODELS_USED:
            s, p = net_ci(VS_FTS[m], f"FTS_{m}", r)
            row.update({f"net_bads_avoided_FTS_{m}_over_noFTS_{m}_CI": s, f"P(delta <= 0) FTS {m} vs no-FTS {m}": p})
    SUMMARY.append(row)
SUMMARY = pd.DataFrame(SUMMARY)

REG_F = FTSR["reg"] if FTS_ON else {}
META = [("module", "s06b_cutoff_analysis, no-FTS variant (spec 8.2, 8.3, 9.1-9.3; E13 scoring code). REPORT ONLY."),
        ("caveat", CAVEAT), ("run_time", RUN_TIME), ("git_commit", GIT_COMMIT), ("config_sha256", CONFIG_HASH),
        ("feature_dict_sha256", FDICT_SHA), ("registry (no-FTS)", REG_PATH), ("registry version / locked_at", f"{REG['version']} / {REG['locked_at']}"),
        ("registry sha256", REG_SHA), ("sign-off", f"{REG['signoff']['decision_rules_signed_by']} {REG['signoff']['signed_date']}"),
        ("locked files hash-verified", len(FILES)), ("Platt file / sha256", f"{PLATT_PATH} / {PLATT_SHA}"),
        ("models", ", ".join(f"{m}: Platt a={PLATT[m]['a']:.4f}, b={PLATT[m]['b']:.4f}" for m in MODELS_USED)),
        ("FTS comparison registry (read-only)", (f"{FTS_REG_PATH} | v{REG_F['version']} locked {REG_F['locked_at']} | sha256 {FTSR['reg_sha']} | "
                                                  f"{len(FTSR['files'])} locked files hash-verified | Platt {FTSR['platt_sha']} | "
                                                  + ", ".join(f"{m}: a={FTSR['platt'][m]['a']:.4f}, b={FTSR['platt'][m]['b']:.4f}" for m in MODELS_USED))
                                                 if FTS_ON else "disabled"),
        ("score scaling", json.dumps(cfg["scaling"])),
        ("input", f"{MASTER_TBL} (Delta v{PROTECTED_TABLES.get(MASTER_TBL)}), approved applicants only, via common.load_window"),
        ("approval-rate base", f"sample_status in {BASE_STATUSES} (config.cutoff_analysis.approval_base_statuses); EXCLUDED not used"),
        ("bad rates", f"{LABELLED} only (config.cutoff_analysis.bad_rate_statuses); INDETERMINATE, IMMATURE, EXCLUDED_PERF reported as separate counts"),
        ("data windows", json.dumps(cfg["windows"])),
        ("OOT window", f"{cfg['windows']['OOT']} on {DATE_A}; base {len(OOT)} = {OOT_COUNTS.to_dict()}; splits {SPLITS}"),
        ("OOT vs spec 1.2 confirmed counts", f"GOOD+BAD {int(OOT_COUNTS['GOOD'] + OOT_COUNTS['BAD'])} / bads {int(OOT_COUNTS['BAD'])} vs config.expected_counts "
                                              f"{cfg.get('expected_counts', {}).get('OOT')} (same rows as E13)"),
        ("DEV second view", f"oof_predictions.parquet p_oof (models A, C) + Platt; base = DEV GOOD+BAD ({len(DEVO['A'])}); other statuses have no OOF prediction; no in-sample DEV scores"),
        ("FWD / STRESS", f"score distribution only (no labels): FWD {len(FWD)}, STRESS {len(STRESS)}, same base statuses"),
        ("cut-off rule", "approve if score >= cut-off; cut-off = highest integer score with approval rate >= target; ties reported"),
        ("bootstrap", f"{BOOT}, stratified on GOOD/BAD, cut-offs fixed; comparisons paired (same resamples for both scores)"),
        ("by_quarter", "full-OOT cut-offs applied to each quarter"),
        ("agreement deciles", "rank deciles of calibrated PD (benchmarks: -score) on the comparison population (1 = lowest risk)"),
        ("segments", f"WATCH only {WATCH_COLS}; config.columns.segments {SEG_MISSING} not in acard_master (open question)"),
        ("has_fts_data", f"Y if {FTS_FLAG_COL} non-null and non-empty after trim; segment only, never a feature"),
        ("benchmark app_id", "CRNNO as stored, or CREDITCARDNO where the normalised CRNNO is repeated in acard_master (s00_build_a_score_snapshot rule)")]
for _, b in BENCH_SRC.iterrows():
    k = re.sub(r"\W", "_", str(b["benchmark"])).upper()
    META += [(f"benchmark {b['benchmark']}", f"{b['description']} | table {b['table']} | source_table {b['source_table']} | column {b['column']} | "
                                            f"join {b['join_key']} | score months used {b['score_month_offsets_used']} | higher_is_lower_risk {b['higher_is_lower_risk']} | "
                                            f"compared on {len(BR[k]['F'])} of {len(OOT)} OOT applicants ({len(BR[k]['F']) / len(OOT):.1%}); no PD, no DEV view"),
             (f"benchmark {b['benchmark']} definition", str(b["definition"]))]
META += [("report-only constants", f"SEG_MIN_BADS {SEG_MIN_BADS}; AGE_BANDS {AGE_BANDS}; TOP_N_WATCH {TOP_N_WATCH}; IMMATURE flags ratio {IMM_RATIO_FLAG}, min n {IMM_BAND_MIN_N}, p {IMM_P_FLAG}")]
META = pd.DataFrame(META, columns=["key", "value"])

SHEETS = {"meta": [("Run metadata", META), ("Benchmarks", BENCH_SRC), ("Status counts in acard_master by window", STATUS_COUNTS), ("E13 spot checks", SPOT)],
          "summary": SUMMARY}
for m in MODELS_USED:
    SHEETS[f"score_bands_{m}"] = BANDS[m]
for m in MODELS_USED:
    SHEETS[f"grid_oot_{m}"] = GRID_OOT[m]
for m in MODELS_USED:
    SHEETS[f"grid_dev_oof_{m}"] = GRID_DEV[m]
SHEETS["C_vs_A"] = C_VS_A
vb = [("Benchmark sources", BENCH_SRC), ("Coverage (score used = joined and inside the allowed score months)", BENCH_COV),
      ("Score month minus application month", BENCH_TIMING), ("Direction check: bad rate by benchmark decile (OOT GOOD/BAD)", BENCH_DIR),
      ("Direction check summary", BENCH_DIR_SUMM), ("Each no-FTS model vs each benchmark", VS_BENCH)]
for k, R in BR.items():
    vb += [(f"Grid on OOT with {R['name']}: {m if m != k else R['name']}", R["GRID"][m]) for m in MODELS_USED + [k]]
    vb += [(f"Score bands {R['name']} (OOT with {R['name']})", R["BANDS"])]
vb += [(f"By quarter: {t}", d) for t, d in QVB]
SHEETS["vs_benchmarks"] = vb
if FTS_ON:
    SHEETS["vs_fts"] = ([(f"FTS {m} vs no-FTS {m} (net bads avoided by FTS)", VS_FTS[m]) for m in MODELS_USED]
                        + [("By has_fts_data (segment only, full-OOT cut-offs)", FTS_SEG), ("Swap sets by has_fts_data", FTS_SEG_SWAP)]
                        + [(f"Grid OOT FTS {m}", GRID_FTS[m]) for m in MODELS_USED] + [(f"By quarter: {t}", d) for t, d in QFTS])
SHEETS.update({
    "swap_sets": [("C vs A (no-FTS), OOT", SWAP)] + [(f"No-FTS models vs {R['name']}", R["SWAP"]) for R in BR.values()]
                 + ([("FTS vs no-FTS, OOT", FTS_SWAP)] if FTS_ON else []),
    "agreement": [("Score agreement", AG)] + AG_CT,
    "by_quarter": [("Score bands by quarter", QB), ("Approval-rate grid by quarter (full-OOT cut-offs)", QG),
                   ("C vs A by quarter", QC), ("Swap sets by quarter", QS), ("Agreement by quarter", QA)] + QCT
                  + [(f"Benchmarks: {t}", d) for t, d in QVB] + [(f"FTS: {t}", d) for t, d in QFTS],
    "immature_profile": [("Flags", IMM_FLAGS), ("By OOT quarter", IMM_Q), ("By application month (gap = APPROVEDATE - APPRECDDATE, days)", IMM_MONTH),
                         ("Gap application -> approval, IMMATURE vs other OOT", IMM_GAP), ("By approval month", IMM_APPROVE),
                         ("By score band", IMM_BAND)],
    "recent_distribution": [("Cut-offs per model (from grid_oot)", pd.DataFrame(CUTS).reset_index(names="approval_rate_target")),
                            ("Score distribution vs OOT (diagnosis only, no labels)", REC)],
    "segments": [("Segment columns (config.columns.segments)", SEG_AVAIL), ("By WATCH variable at each cut-off", SEGROWS)]})
save_excel(SHEETS, XLSX_PATH)

# Open question requested by the owner: append once, replace the line on re-runs (idempotent)
TAG = "[S06b-segments]"
line = (f"- {TAG} s06b_cutoff_analysis ({RUN_TIME[:10]}, {VERSION}): config.columns.segments {SEG_MISSING} are not columns of {MASTER_TBL}. "
        f"The cut-off segment analysis uses the WATCH variables only ({', '.join(WATCH_COLS)}). Should these segment columns be added "
        f"to acard_master (and from which source), or removed from config.columns.segments?")
old = open(OQ_PATH).read() if os.path.exists(OQ_PATH) else "# Open questions\n"
lines = [l for l in old.rstrip("\n").split("\n") if TAG not in l]
if SEG_MISSING:
    lines += ["", line] if lines and lines[-1].strip() else [line]
write_text("\n".join(lines) + "\n", OQ_PATH)

import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="s06b_cutoff_analysis_nofts") as run:
    mlflow.log_params({"module": "s06b_cutoff_analysis", "registry_version": REG["version"], "models": ",".join(MODELS_USED),
                       "approval_rate_grid": ",".join(map(str, GRID)), "score_band_width": BW, "base_statuses": ",".join(BASE_STATUSES),
                       "config_sha256": CONFIG_HASH, "git_commit": GIT_COMMIT, "registry_sha256": REG_SHA,
                       "fts_registry": f"{REG_F.get('version')} {FTSR['reg_sha'] if FTS_ON else ''}"})
    for m in MODELS_USED:
        for _, g in GRID_OOT[m].iterrows():
            kk = f"{int(round(g['approval_rate_target'] * 1000))}"
            mlflow.log_metrics({f"{m}_cutoff_{kk}": g["cutoff_score"], f"{m}_approved_bad_rate_{kk}": g["approved_bad_rate"],
                                f"{m}_share_bads_declined_{kk}": g["share_of_bads_declined"]})
    for _, c in C_VS_A.iterrows():
        mlflow.log_metric(f"net_bads_avoided_C_vs_A_{int(round(c['approval_rate_target'] * 1000))}", c["net_bads_avoided_by_C"])
    for k, R in BR.items():
        for x in MODELS_USED:
            for _, v in R["VS"][x].iterrows():
                mlflow.log_metric(f"net_bads_avoided_{x}_vs_{k.lower()}_{int(round(v['approval_rate_target'] * 1000))}", v[f"net_bads_avoided_by_{x}"])
    for m, t in VS_FTS.items():
        for _, v in t.iterrows():
            mlflow.log_metric(f"net_bads_avoided_FTS_{m}_over_noFTS_{int(round(v['approval_rate_target'] * 1000))}", v[f"net_bads_avoided_by_FTS_{m}"])
    mlflow.log_artifact(XLSX_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"Workbook: {XLSX_PATH} ({len(SHEETS)} sheets) | MLflow run {MLFLOW_RUN_ID}")
display(SUMMARY)

In [0]:
import openpyxl
CHECKS = {}
# 1. Registries, locked files, calibration unchanged; nothing written outside the output folder
for R in [NF] + ([FTSR] if FTS_ON else []):
    lb = R["label"]
    CHECKS[f"{lb} registry file unchanged"] = sha256_file(R["path"]) == R["reg_sha"]
    CHECKS[f"{lb}: all locked model/binning/fold/OOF file hashes match the registry"] = all(
        sha256_file(v["path"]) == v["sha256"] for v in read_json(R["path"])["files"].values())
    CHECKS[f"{lb}: Platt calibration file unchanged (registry hash)"] = sha256_file(R["platt_path"]) == R["platt_sha"] == R["cal_ref"]["sha256"]
CHECKS["no file created/changed under the no-FTS model folder outside the output folder"] = tree_snapshot(MODEL_ROOT) == SNAP0
if FTS_ON:
    CHECKS["no file created/changed under the FTS model folder"] = tree_snapshot(FTS_ROOT) == SNAP0_FTS
CHECKS["protected Delta tables unchanged (acard_master, feature logs, E13 predictions, benchmark tables)"] = {t: table_version(t) for t in PROTECTED_TABLES} == PROTECTED_TABLES
CHECKS["every file written is inside the output folder (except the requested open_questions.md entry)"] = all(
    os.path.abspath(p).startswith(os.path.abspath(OUT) + os.sep) or os.path.abspath(p) == os.path.abspath(OQ_PATH) for p in WRITTEN) and os.path.exists(XLSX_PATH)
CHECKS["nothing written under main_artifacts_root or the FTS model folder"] = not any(
    os.path.abspath(p).startswith(os.path.abspath(r) + os.sep) for p in WRITTEN for r in [MAIN_ROOT, FTS_ROOT] if r)

# 2. Benchmark tables: one row per join_key (re-checked with SQL)
for b in BENCHES:
    if b.get("table"):
        r_ = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT {b['join_key']}) AS d FROM {b['table']}").first()
        CHECKS[f"benchmark {b['name']}: one row per {b['join_key']} in {b['table']}"] = bool(BENCH_UNIQUE[b["key"]] and r_["n"] == r_["d"])

# 3. OOT scores reproduce the stored E13 scores (1,000 rows) for every no-FTS and FTS model
for _, rw in SPOT.iterrows():
    CHECKS[f"{rw['registry']} {rw['model']}: E13 spot check (1,000 rows, exact score, PD within 1e-9)"] = bool(
        rw["n"] == 1000 and rw["n_score_diff"] == 0 and rw["max_abs_pd_diff"] <= 1e-9)

# 4. Reconciliation for every model and cut-off
G_ALL = pd.concat([*GRID_OOT.values(), *GRID_DEV.values(), QG, *GRID_FTS.values()]
                  + [g for R in BR.values() for g in R["GRID"].values()] + [d for t, d in QVB if "grid" in t], ignore_index=True)
CHECKS["grids: approved + declined = approval base (every model, cut-off, view)"] = bool((G_ALL["n_approved"] + G_ALL["n_declined"] == G_ALL["n_total"]).all())
CHECKS["grids: bads approved + bads declined = total bads"] = bool((G_ALL["bads_approved"] + G_ALL["bads_declined"] == G_ALL["bads_total"]).all())
CHECKS["grids: GOOD+BAD approved + declined = total GOOD+BAD"] = bool((G_ALL["good_bad_approved"] + G_ALL["good_bad_declined"] == G_ALL["good_bad_total"]).all())
CHECKS["OOT grids (no-FTS and FTS): total = OOT approval base and total bads = OOT BAD"] = all(
    (g["n_total"] == len(OOT)).all() and (g["bads_total"] == OOT_COUNTS["BAD"]).all() for g in [*GRID_OOT.values(), *GRID_FTS.values()])
CHECKS["OOT grids: other statuses approved + declined = their OOT counts"] = all(
    (g[f"{s}_approved"] + g[f"{s}_declined"] == OOT_COUNTS[s]).all() for g in [*GRID_OOT.values(), *GRID_FTS.values()] for s in OTHER)
CHECKS["score bands reconcile to the OOT base and bads"] = all(
    t["n"].sum() == len(OOT) and t["BAD"].sum() == OOT_COUNTS["BAD"] and t["cum_bads_declined"].iloc[-1] == 0 for t in BANDS.values())
sw = pd.concat([SWAP, QS]).groupby(["split", "approval_rate_target"])["n"].sum()
split_n = {"OOT": len(OOT), **OOT["oot_split"].value_counts().to_dict()}
CHECKS["swap sets C vs A: four groups reconcile to the population (OOT and each quarter)"] = all(v == split_n[s] for (s, _), v in sw.items())
for k, R in BR.items():
    CHECKS[f"benchmark {R['name']}: comparison population = OOT rows with the benchmark, same rows for every score"] = bool(
        len(R["F"]) == int(OOT[f"has_{k}"].sum()) and all((g["n_total"] == len(R["F"])).all() for g in R["GRID"].values()))
    CHECKS[f"benchmark {R['name']}: swap sets reconcile to the comparison population"] = bool(
        (R["SWAP"].groupby(["pair", "approval_rate_target"])["n"].sum() == len(R["F"])).all())
    off = next(b for b in BENCHES if b["key"] == k).get("table") and BENCH_SRC.set_index("benchmark").loc[R["name"], "date_column"]
    if off:
        CHECKS[f"benchmark {R['name']}: every score used has score month - application month in {ASCORE_OFFSETS}"] = bool(
            all(df.loc[df[f"has_{k}"], f"bench_month_offset_{k}"].isin(ASCORE_OFFSETS).all() for df in (OOT, FWD, STRESS)))
if FTS_ON:
    CHECKS["FTS: swap sets reconcile to the OOT base"] = bool((FTS_SWAP.groupby(["pair", "approval_rate_target"])["n"].sum() == len(OOT)).all())
    seg_n = OOT["has_fts_data"].value_counts().to_dict()
    CHECKS["FTS: has_fts_data segments reconcile (Y + N = OOT base; swap groups = segment n)"] = bool(
        sum(seg_n.values()) == len(OOT) and all(v == seg_n[s.split("=")[1][0]] for (s, _, _), v in
                                                FTS_SEG_SWAP.groupby(["split", "pair", "approval_rate_target"])["n"].sum().items()))
CHECKS["segments: approved + declined = segment n; bads approved + declined = segment bads"] = bool(
    ((SEGROWS["n_approved"] + SEGROWS["n_declined"]) == SEGROWS["n"]).all() and ((SEGROWS["bads_approved"] + SEGROWS["bads_declined"]) == SEGROWS["bads"]).all())
_segn = SEGROWS.groupby(["dim", "model", "approval_rate_target"])["n"].sum().reset_index()
_exp = {m: len(OOT) for m in MODELS_USED} | {BENCH_NAME_OF[k]: len(BR[k]["F"]) for k in BENCH_KEYS}
CHECKS["segments: each WATCH dimension covers the OOT base (benchmarks: OOT with the benchmark)"] = bool((_segn["n"] == _segn["model"].map(_exp)).all())
CHECKS["benchmarks not used to score the models (model scores = scaling of their own PD)"] = all(
    (OOT[f"score_{p}{m}"].to_numpy() == common.pd_to_score(OOT[f"pd_{p}{m}"].to_numpy(), cfg)).all() for m in MODELS_USED for p in [""] + (["FTS_"] if FTS_ON else []))

# 5. Bad rates use OOT GOOD/BAD or DEV OOF only; no in-sample DEV scores
CHECKS["no DEV rows scored with the models (DEV only via OOF predictions)"] = "DEV" not in SCORED_WINDOWS and set(OOT["window"]) == {"OOT"}
CHECKS["DEV view PD = Platt(p_oof) for every row"] = all(
    np.allclose(DEVO[m][f"pd_{m}"], common.apply_platt(DEVO[m]["p_oof"].to_numpy(), PLATT[m]), atol=0, rtol=1e-12) for m in MODELS_USED)
_chk = []                                                        # independent recomputation with pandas filters
for m in MODELS_USED:
    for r in (GRID[0], GRID[-1]):
        c = CUTS[m][r]
        lab = OOT[OOT["sample_status"].isin(["GOOD", "BAD"]) & (OOT[f"score_{m}"] >= c)]
        _chk.append(np.isclose((lab["sample_status"] == "BAD").mean(),
                               GRID_OOT[m].set_index("approval_rate_target").loc[r, "approved_bad_rate"]))
CHECKS["approved bad rate = BAD / (GOOD + BAD) among approved (independent recomputation)"] = all(_chk)

# 6. Workbook: required sheets and the approved-only caveat on every sheet
wb = openpyxl.load_workbook(XLSX_PATH, read_only=True)
req = ["meta", "score_bands_A", "score_bands_C", "grid_oot_A", "grid_oot_C", "grid_dev_oof_A", "grid_dev_oof_C", "C_vs_A",
       "vs_benchmarks", "swap_sets", "agreement", "by_quarter", "immature_profile", "recent_distribution", "segments", "summary"] + (["vs_fts"] if FTS_ON else [])
CHECKS["workbook has every required sheet"] = all(s in wb.sheetnames for s in req)
CHECKS["approved-only caveat in cell A1 of every sheet"] = all(next(wb[s].iter_rows(min_row=1, max_row=1, values_only=True))[0] == CAVEAT for s in wb.sheetnames)
wb.close()

RES = pd.DataFrame([{"check": k, "pass": bool(v)} for k, v in CHECKS.items()])
display(RES)
ALL_PASS = bool(RES["pass"].all())
write_json({"module": "s06b_cutoff_analysis", "variant": VERSION, "run_time": RUN_TIME, "git_commit": GIT_COMMIT, "config_sha256": CONFIG_HASH,
            "registry": {"path": REG_PATH, "version": REG["version"], "sha256": REG_SHA},
            "fts_registry": {"path": FTS_REG_PATH, "version": REG_F.get("version"), "sha256": FTSR["reg_sha"] if FTS_ON else None},
            "rows": {"oot_base": len(OOT), "oot_by_status": OOT_COUNTS.to_dict(), "fwd": len(FWD), "stress": len(STRESS),
                     "dev_oof": {m: len(d) for m, d in DEVO.items()}},
            "benchmark_coverage_oot": BENCH_COV[BENCH_COV["window"] == "OOT"].to_dict("records"),
            "cutoffs": {m: {str(k): int(v) for k, v in CUTS[m].items()} for m in MODELS_USED},
            "approved_bad_rate": {m: dict(zip(map(str, GRID_OOT[m]["approval_rate_target"]), GRID_OOT[m]["approved_bad_rate"])) for m in MODELS_USED},
            "net_bads_avoided_C_vs_A": dict(zip(map(str, C_VS_A["approval_rate_target"]), C_VS_A["net_bads_avoided_by_C"])),
            "net_bads_avoided_FTS_over_noFTS": {m: dict(zip(map(str, t["approval_rate_target"]), t[f"net_bads_avoided_by_FTS_{m}"])) for m, t in VS_FTS.items()},
            "outputs": WRITTEN + [SUMMARY_PATH], "mlflow_run_id": MLFLOW_RUN_ID,
            "checks": CHECKS, "all_pass": ALL_PASS, "runtime_min": round((time.time() - _t0) / 60, 2)}, SUMMARY_PATH)
failed = RES.loc[~RES["pass"], "check"].tolist()
assert ALL_PASS, f"Acceptance FAILED: {failed}"
print(f"All {len(RES)} acceptance checks PASS | run summary {SUMMARY_PATH}")